# 01 – Data Understanding

**Project:** Loan Default Prediction (LendingClub accepted loans)

**Purpose of this notebook:** understand the structure, quality and meaning of the raw data; decide which columns are legitimate model inputs; and record the cleaning actions to be carried out in the next stage.

**Prediction point:** application time, i.e. before the lender evaluates the application. Only information available at that point may be used as a model input. Anything generated after the application is evaluated or the loan is issued is treated as potential data leakage.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", None)

## 2. Data Loading

In [ ]:
DATA_PATH = Path("..") / "data" / "interim" / "accepted_2007_to_201804.parquet"
df = pd.read_parquet(DATA_PATH)

## 3. Initial Inspection

In [ ]:
df.shape  # (2260701, 151)

In [ ]:
df.info()

In [ ]:
print(df.dtypes.to_string())

### Preview of the data

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.sample(5, random_state=42)

### Distribution overview

With 151 columns, summary statistics alone are hard to interpret. Distributions will be examined graphically in the EDA notebook.

In [ ]:
df.describe().T

## 4. Categorical Columns: Cardinality and Missing Values

The columns below are likely to be categorical, based on their data type and meaning.

In [ ]:
categorical_cols = [
    "term",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "verification_status",
    "loan_status",
    "pymnt_plan",
    "purpose",
    "initial_list_status",
    "application_type",
    "verification_status_joint",
    "hardship_flag",
    "hardship_type",
    "hardship_reason",
    "hardship_status",
    "hardship_loan_status",
    "disbursement_method",
    "debt_settlement_flag",
    "settlement_status",
    "addr_state",
    "zip_code"
]

In [ ]:
categorical_summary = pd.DataFrame({
    "unique_values": df[categorical_cols].nunique(),
    "missing_%": df[categorical_cols].isna().mean().mul(100).round(5),
}).sort_values("missing_%", ascending=False)

categorical_summary

### Missing-value assessment

**Very high missingness (approximately 94% or more):** `verification_status_joint`, `hardship_type`, `hardship_reason`, `hardship_status`, `hardship_loan_status`, `settlement_status`.
- This missingness is structural rather than random. A value exists only when the event applies (a joint application, a hardship plan, a debt settlement), so a missing value means "not applicable".
- The `hardship_*` and `settlement_*` columns describe events after loan origination. They are excluded because of their timing (see Section 6), regardless of their missingness.
- `verification_status_joint` has the same structural missingness as the other joint-application columns. It is therefore handled together with that group (see Section 6) rather than dropped on missingness alone.

**Low missingness:** apart from `emp_length` and the columns above, the categorical columns have missing values in only 34 rows (0.0015% of the dataset).
- The proportion is negligible, so these rows can be removed instead of imputing values.
- The rows should be inspected before removal to confirm that they are blank or non-loan records.

**`emp_length`:** investigated separately because its missingness is much higher (6.5%).

In [ ]:
low_null_cols = [
    "term",
    "grade",
    "sub_grade",
    "home_ownership",
    "verification_status",
    "loan_status",
    "pymnt_plan",
    "purpose",
    "initial_list_status",
    "application_type",
    "hardship_flag",
    "disbursement_method",
    "debt_settlement_flag",
    "addr_state",
    "zip_code"
]

rows_with_missing = df[low_null_cols].isna().any(axis=1)

print(f"Rows affected: {rows_with_missing.sum()}")
print(f"Percentage affected: {rows_with_missing.mean() * 100:.4f}%")

### `emp_length`

In [ ]:
emp_length_share = df["emp_length"].value_counts(dropna=False, normalize=True).mul(100).round(2)
emp_length_share

In [ ]:
emp_length_order = ["< 1 year", "1 year"] + [f"{i} years" for i in range(2, 10)] + ["10+ years", "Missing"]

emp_length_counts = df["emp_length"].fillna("Missing").value_counts()
emp_length_order = (
    [v for v in emp_length_order if v in emp_length_counts.index]
    + [v for v in emp_length_counts.index if v not in emp_length_order]
)

plt.figure(figsize=(10, 4))
sns.barplot(x=emp_length_counts.reindex(emp_length_order).index,
            y=emp_length_counts.reindex(emp_length_order).values)
plt.xlabel("emp_length")
plt.ylabel("Number of loans")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

- `emp_length` has 6.5% missing values. This is too large a share to drop without further investigation.
- The missing values will be kept as a separate category for now. Missingness may itself be informative (for example, when employment length is not reported), so its relationship with `loan_status` will be examined in the EDA before the final treatment is decided.

## 5. Domain Notes

- **Credit inquiries (`inq_last_6mths`, `inq_last_12m`):** lenders commonly treat more than three to four hard inquiries within six months as a warning sign, because it may indicate that the borrower is seeking several lines of credit at once.
- **FICO score:** a three-digit credit score between 300 and 850 that estimates how likely a borrower is to repay a loan. FICO reports that its scores are used by about 90% of top US lenders. In this dataset the score is given as a range (`fico_range_low`, `fico_range_high`) recorded at origination.
- **`delinq_2yrs`:** the number of 30+ days past-due delinquency incidences in the borrower's credit file during the previous two years.

## 6. Column Significance at the Prediction Point

**Principle:** the model may use only information that is legitimately available at the application-time prediction point. Information generated after the application is evaluated or the loan is issued must not be used as an input feature.

Each column is assigned to a category below, and each category receives a decision.

In [ ]:
column_categories = {
    "Application Information": [
        "loan_amnt",
        "term",
        "emp_length",
        "home_ownership",
        "annual_inc",
        "verification_status",
        "purpose",
        "zip_code",
        "addr_state",
        "dti",
        "application_type",
    ],

    "Credit History": [
        "delinq_2yrs",
        "earliest_cr_line",
        "fico_range_low",
        "fico_range_high",
        "inq_last_6mths",
        "mths_since_last_delinq",
        "mths_since_last_record",
        "open_acc",
        "pub_rec",
        "revol_bal",
        "revol_util",
        "total_acc",
        "collections_12_mths_ex_med",
        "mths_since_last_major_derog",
        "acc_now_delinq",
        "tot_coll_amt",
        "tot_cur_bal",
        "open_acc_6m",
        "open_act_il",
        "open_il_12m",
        "open_il_24m",
        "mths_since_rcnt_il",
        "total_bal_il",
        "il_util",
        "open_rv_12m",
        "open_rv_24m",
        "max_bal_bc",
        "all_util",
        "total_rev_hi_lim",
        "inq_fi",
        "total_cu_tl",
        "inq_last_12m",
        "acc_open_past_24mths",
        "avg_cur_bal",
        "bc_open_to_buy",
        "bc_util",
        "chargeoff_within_12_mths",
        "delinq_amnt",
        "mo_sin_old_il_acct",
        "mo_sin_old_rev_tl_op",
        "mo_sin_rcnt_rev_tl_op",
        "mo_sin_rcnt_tl",
        "mort_acc",
        "mths_since_recent_bc",
        "mths_since_recent_bc_dlq",
        "mths_since_recent_inq",
        "mths_since_recent_revol_delinq",
        "num_accts_ever_120_pd",
        "num_actv_bc_tl",
        "num_actv_rev_tl",
        "num_bc_sats",
        "num_bc_tl",
        "num_il_tl",
        "num_op_rev_tl",
        "num_rev_accts",
        "num_rev_tl_bal_gt_0",
        "num_sats",
        "num_tl_120dpd_2m",
        "num_tl_30dpd",
        "num_tl_90g_dpd_24m",
        "num_tl_op_past_12m",
        "pct_tl_nvr_dlq",
        "percent_bc_gt_75",
        "pub_rec_bankruptcies",
        "tax_liens",
        "tot_hi_cred_lim",
        "total_bal_ex_mort",
        "total_bc_limit",
        "total_il_high_credit_limit",
    ],

    "Joint Application / Credit History": [
        "annual_inc_joint",
        "dti_joint",
        "verification_status_joint",
        "revol_bal_joint",
        "sec_app_fico_range_low",
        "sec_app_fico_range_high",
        "sec_app_earliest_cr_line",
        "sec_app_inq_last_6mths",
        "sec_app_mort_acc",
        "sec_app_open_acc",
        "sec_app_revol_util",
        "sec_app_open_act_il",
        "sec_app_num_rev_accts",
        "sec_app_chargeoff_within_12_mths",
        "sec_app_collections_12_mths_ex_med",
        "sec_app_mths_since_last_major_derog",
    ],

    "Lender-Assigned / Listing Attributes": [
        "int_rate",
        "installment",
        "grade",
        "sub_grade",
        "initial_list_status",
    ],

    "Funding / Loan Issuance": [
        "funded_amnt",
        "funded_amnt_inv",
        "issue_d",
        "disbursement_method",
    ],

    "Post-Loan Performance": [
        "out_prncp",
        "out_prncp_inv",
        "total_pymnt",
        "total_pymnt_inv",
        "total_rec_prncp",
        "total_rec_int",
        "total_rec_late_fee",
        "recoveries",
        "collection_recovery_fee",
        "last_pymnt_d",
        "last_pymnt_amnt",
        "next_pymnt_d",
        "last_credit_pull_d",
        "last_fico_range_high",
        "last_fico_range_low",
        "pymnt_plan",
    ],

    "Hardship / Payment Modification": [
        "hardship_flag",
        "hardship_type",
        "hardship_reason",
        "hardship_status",
        "deferral_term",
        "hardship_amount",
        "hardship_start_date",
        "hardship_end_date",
        "payment_plan_start_date",
        "hardship_length",
        "hardship_dpd",
        "hardship_loan_status",
        "orig_projected_additional_accrued_interest",
        "hardship_payoff_balance_amount",
        "hardship_last_payment_amount",
    ],

    "Debt Settlement": [
        "debt_settlement_flag",
        "debt_settlement_flag_date",
        "settlement_status",
        "settlement_date",
        "settlement_amount",
        "settlement_percentage",
        "settlement_term",
    ],

    "Identifiers / Administrative": [
        "id",
        "member_id",
        "url",
        "policy_code",
    ],

    "Text / Free Text": [
        "emp_title",
        "title",
        "desc",
    ],

    "Target": [
        "loan_status",
    ],

}

In [ ]:
column_meanings = {
    "id": "Unique identifier assigned to the loan",
    "member_id": "Unique identifier of the borrower/member",
    "loan_amnt": "Amount of money requested by the borrower",
    "funded_amnt": "Amount of the requested loan actually funded",
    "funded_amnt_inv": "Amount of the funded loan contributed by investors",
    "term": "Loan repayment duration in months",
    "int_rate": "Interest rate charged on the loan",
    "installment": "Monthly payment amount scheduled for the loan",
    "grade": "Loan risk grade assigned by the lender",
    "sub_grade": "Detailed risk grade assigned to the loan",
    "emp_title": "Job title or occupation of the borrower",
    "emp_length": "Length of the borrower's employment in years (from under 1 year to 10 or more years)",
    "home_ownership": "Borrower's home ownership status",
    "annual_inc": "Borrower's self-reported annual income",
    "verification_status": "Whether the borrower's income (or its source) was verified by the lender",
    "issue_d": "Date on which the loan was issued",
    "loan_status": "Current or final status/outcome of the loan",
    "pymnt_plan": "Indicates whether a payment plan has been put in place for the loan",
    "url": "URL associated with the loan listing",
    "desc": "Borrower's description or explanation of the loan",
    "purpose": "Purpose for which the borrower requested the loan",
    "title": "Title provided by the borrower for the loan",
    "zip_code": "ZIP code of the borrower's address",
    "addr_state": "State of the borrower's address",
    "dti": "Ratio of the borrower's monthly debt payments (excluding mortgage and the requested loan) to self-reported monthly income",
    "delinq_2yrs": "Number of 30+ days past-due delinquency incidences in the borrower's credit file during the previous two years",
    "earliest_cr_line": "Date when the borrower's oldest credit account was opened",
    "fico_range_low": "Lower bound of the borrower's FICO score range at loan origination",
    "fico_range_high": "Upper bound of the borrower's FICO score range at loan origination",
    "inq_last_6mths": "Number of credit inquiries during the previous six months (excluding auto and mortgage inquiries)",
    "mths_since_last_delinq": "Months since the borrower's most recent delinquency",
    "mths_since_last_record": "Months since the borrower's most recent public record",
    "open_acc": "Number of currently open credit accounts",
    "pub_rec": "Number of derogatory public records",
    "revol_bal": "Total outstanding revolving credit balance",
    "revol_util": "Percentage of revolving credit currently being used",
    "total_acc": "Total number of credit accounts",
    "initial_list_status": "Initial listing status of the loan (values W and F)",
    "out_prncp": "Outstanding principal remaining on the loan",
    "out_prncp_inv": "Outstanding principal remaining for investors",
    "total_pymnt": "Total amount of payments received on the loan",
    "total_pymnt_inv": "Total amount of payments received by investors",
    "total_rec_prncp": "Total principal amount received from the borrower",
    "total_rec_int": "Total interest amount received from the borrower",
    "total_rec_late_fee": "Total late-payment fees received",
    "recoveries": "Amount recovered after the loan was charged off",
    "collection_recovery_fee": "Fee collected from recovery/collection activity",
    "last_pymnt_d": "Date of the most recent payment",
    "last_pymnt_amnt": "Amount of the most recent payment",
    "next_pymnt_d": "Scheduled date of the next payment",
    "last_credit_pull_d": "Date when the borrower's credit information was last pulled",
    "last_fico_range_high": "Upper bound of the most recently recorded FICO range",
    "last_fico_range_low": "Lower bound of the most recently recorded FICO range",
    "collections_12_mths_ex_med": "Number of collections during the previous 12 months excluding medical collections",
    "mths_since_last_major_derog": "Months since the most recent 90-day or worse rating",
    "policy_code": "Lending policy code associated with the loan",
    "application_type": "Whether the application is individual or joint",
    "annual_inc_joint": "Combined annual income reported for a joint application",
    "dti_joint": "Combined debt-to-income ratio for a joint application",
    "verification_status_joint": "Income verification status for a joint application",
    "acc_now_delinq": "Number of accounts currently delinquent",
    "tot_coll_amt": "Total collection amounts ever owed",
    "tot_cur_bal": "Total current balance across credit accounts",
    "open_acc_6m": "Number of credit accounts opened in the recent six-month period",
    "open_act_il": "Number of currently active installment accounts",
    "open_il_12m": "Number of installment accounts opened in the previous 12 months",
    "open_il_24m": "Number of installment accounts opened in the previous 24 months",
    "mths_since_rcnt_il": "Months since the most recently opened installment account",
    "total_bal_il": "Total balance of installment accounts",
    "il_util": "Ratio of total current balance to credit limit across all installment accounts",
    "open_rv_12m": "Number of revolving accounts opened in the previous 12 months",
    "open_rv_24m": "Number of revolving accounts opened in the previous 24 months",
    "max_bal_bc": "Maximum current balance owed across all revolving accounts",
    "all_util": "Overall utilization across credit accounts",
    "total_rev_hi_lim": "Total revolving credit limit",
    "inq_fi": "Number of personal finance inquiries",
    "total_cu_tl": "Total number of finance/trade accounts",
    "inq_last_12m": "Number of credit inquiries during the previous 12 months",
    "acc_open_past_24mths": "Number of accounts opened during the previous 24 months",
    "avg_cur_bal": "Average current balance across credit accounts",
    "bc_open_to_buy": "Available credit on bankcard accounts",
    "bc_util": "Bankcard credit utilization",
    "chargeoff_within_12_mths": "Number of charge-offs during the previous 12 months",
    "delinq_amnt": "Past-due amount owed on accounts on which the borrower is currently delinquent",
    "mo_sin_old_il_acct": "Months since the oldest installment account was opened",
    "mo_sin_old_rev_tl_op": "Months since the oldest revolving account was opened",
    "mo_sin_rcnt_rev_tl_op": "Months since the most recently opened revolving account",
    "mo_sin_rcnt_tl": "Months since the most recently opened credit account",
    "mort_acc": "Number of mortgage accounts",
    "mths_since_recent_bc": "Months since the most recently opened bankcard account",
    "mths_since_recent_bc_dlq": "Months since the most recent bankcard delinquency",
    "mths_since_recent_inq": "Months since the most recent credit inquiry",
    "mths_since_recent_revol_delinq": "Months since the most recent revolving-account delinquency",
    "num_accts_ever_120_pd": "Number of accounts that have ever been 120 or more days past due",
    "num_actv_bc_tl": "Number of currently active bankcard accounts",
    "num_actv_rev_tl": "Number of currently active revolving accounts",
    "num_bc_sats": "Number of satisfactory bankcard accounts",
    "num_bc_tl": "Total number of bankcard accounts",
    "num_il_tl": "Total number of installment accounts",
    "num_op_rev_tl": "Number of currently open revolving accounts",
    "num_rev_accts": "Total number of revolving accounts",
    "num_rev_tl_bal_gt_0": "Number of revolving accounts with a balance greater than zero",
    "num_sats": "Number of satisfactory credit accounts",
    "num_tl_120dpd_2m": "Number of accounts currently 120 or more days past due (updated within the past two months)",
    "num_tl_30dpd": "Number of accounts currently 30 days past due (updated within the past two months)",
    "num_tl_90g_dpd_24m": "Number of accounts 90 or more days past due during the previous 24 months",
    "num_tl_op_past_12m": "Number of accounts opened during the previous 12 months",
    "pct_tl_nvr_dlq": "Percentage of credit accounts that have never been delinquent",
    "percent_bc_gt_75": "Percentage of bankcards using more than 75 percent of their limit",
    "pub_rec_bankruptcies": "Number of public-record bankruptcies",
    "tax_liens": "Number of tax liens",
    "tot_hi_cred_lim": "Total high credit limit across credit accounts",
    "total_bal_ex_mort": "Total credit balance excluding mortgage accounts",
    "total_bc_limit": "Total credit limit across bankcard accounts",
    "total_il_high_credit_limit": "Total high credit limit across installment accounts",
    "revol_bal_joint": "Combined revolving credit balance for joint applicants",
    "sec_app_fico_range_low": "Lower bound of the secondary applicant's FICO score range",
    "sec_app_fico_range_high": "Upper bound of the secondary applicant's FICO score range",
    "sec_app_earliest_cr_line": "Date when the secondary applicant's oldest credit account was opened",
    "sec_app_inq_last_6mths": "Secondary applicant's credit inquiries during the previous six months",
    "sec_app_mort_acc": "Number of mortgage accounts held by the secondary applicant",
    "sec_app_open_acc": "Number of open credit accounts held by the secondary applicant",
    "sec_app_revol_util": "Revolving credit utilization of the secondary applicant",
    "sec_app_open_act_il": "Number of active installment accounts held by the secondary applicant",
    "sec_app_num_rev_accts": "Number of revolving accounts held by the secondary applicant",
    "sec_app_chargeoff_within_12_mths": "Secondary applicant's charge-offs during the previous 12 months",
    "sec_app_collections_12_mths_ex_med": "Secondary applicant's collections during the previous 12 months excluding medical collections",
    "sec_app_mths_since_last_major_derog": "Months since the secondary applicant's most recent major derogatory event",
    "hardship_flag": "Indicates whether the borrower entered a hardship program",
    "hardship_type": "Type of hardship assistance provided",
    "hardship_reason": "Reason for the hardship program",
    "hardship_status": "Current status of the hardship program",
    "deferral_term": "Number of months for which payments were deferred",
    "hardship_amount": "Monthly interest payment the borrower committed to make during the hardship plan",
    "hardship_start_date": "Date the hardship plan started",
    "hardship_end_date": "Date the hardship plan ended",
    "payment_plan_start_date": "Date the hardship payment plan started",
    "hardship_length": "Duration of the hardship program",
    "hardship_dpd": "Account days past due as of the start date of the hardship plan",
    "hardship_loan_status": "Loan status while under the hardship program",
    "orig_projected_additional_accrued_interest": "Projected additional interest resulting from the hardship plan",
    "hardship_payoff_balance_amount": "Payoff balance associated with the hardship plan",
    "hardship_last_payment_amount": "Last payment amount as of the start date of the hardship plan",
    "disbursement_method": "Method used to disburse the loan funds",
    "debt_settlement_flag": "Indicates whether a charged-off borrower is working with a debt-settlement company",
    "debt_settlement_flag_date": "Date the debt settlement flag was recorded",
    "settlement_status": "Status of the debt settlement process",
    "settlement_date": "Date the debt settlement occurred",
    "settlement_amount": "Amount agreed upon in the debt settlement",
    "settlement_percentage": "Settlement amount as a percentage of the payoff balance of the loan",
    "settlement_term": "Duration of the debt settlement arrangement",
}

In [ ]:
column_reference_df = pd.DataFrame(
    [
        {
            "Column": col,
            "What it represents": column_meanings[col],
            "Type / stage": category,
        }
        for category, columns in column_categories.items()
        for col in columns
    ]
)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(column_reference_df)

| Category | Understanding | Initial decision |
| --- | --- | --- |
| **Application Information** | Information provided by the borrower during the loan application | **Include as candidate features** |
| **Credit History** | Credit-bureau information available when the borrower is evaluated | **Include as candidate features** |
| **Joint Application / Credit History** | Additional applicant and credit information available only for joint applications | **Include as candidate features when applicable** |
| **Lender-Assigned / Listing Attributes** | Terms and attributes produced by the lender's own risk assessment and listing process (`grade`, `sub_grade`, `int_rate`, `installment`, `initial_list_status`) | **Exclude from application-time model** |
| **Funding / Loan Issuance** | Information generated when the loan is funded or issued | **Exclude from application-time model** |
| **Post-Loan Performance** | Information generated by repayment behaviour after issuance | **Exclude from application-time model** |
| **Hardship / Payment Modification** | Information generated when a borrower enters a hardship or payment-modification plan after origination | **Exclude from application-time model** |
| **Debt Settlement** | Information generated by later debt-settlement activity | **Exclude from application-time model** |
| **Identifiers / Administrative** | IDs, URLs and administrative fields used mainly for identification and management | **Investigate, likely exclude** |
| **Text / Free Text** | Free-text fields: employment title, loan title and borrower description | **Investigate separately** |
| **Target (`loan_status`)** | Current or final loan outcome, used to define the prediction target | **Target, not a feature** |

**Lender-assigned attributes:** `grade`, `sub_grade` and `int_rate` are the output of the lender's own risk assessment, and `installment` is calculated from `int_rate`. They are not known before the application is evaluated, and using them would largely reproduce the lender's existing credit decision. They are excluded from the application-time model. If the prediction point is redefined as "after the loan offer", they can be reconsidered.

## 7. Identifiers / Administrative Columns

In [ ]:
identifier_cols = column_categories["Identifiers / Administrative"]

identifier_info = pd.DataFrame({
    "Column": identifier_cols,
    "Data_Type": df[identifier_cols].dtypes.astype(str).values,
    "Null_%": df[identifier_cols].isna().mean().mul(100).values,
    "Unique_Values": df[identifier_cols].nunique(dropna=True).values,
    "Unique_%": (df[identifier_cols].nunique(dropna=True) / len(df) * 100).values,
})

identifier_info

`id`, `member_id`, `url` and `policy_code` were checked for missingness, uniqueness and variability. They carry no borrower-level predictive information for the application-time model and are **excluded**.

## 8. Text / Free-Text Columns

In [ ]:
text_cols = column_categories["Text / Free Text"]

text_info = pd.DataFrame({
    "Column": text_cols,
    "Data_Type": df[text_cols].dtypes.astype(str).values,
    "Null_%": df[text_cols].isna().mean().mul(100).values,
    "Unique_Values": df[text_cols].nunique(dropna=True).values,
    "Unique_%": (df[text_cols].nunique(dropna=True) / len(df) * 100).values,
})

text_info

In [ ]:
for col in text_cols:
    print(f"\n{'=' * 50}")
    print(f"Column: {col}")
    print(f"{'=' * 50}")

    print("Sample values:")
    print(df[col].dropna().sample(min(5, df[col].notna().sum()), random_state=42).tolist())

    print("Average text length:", df[col].dropna().astype(str).str.len().mean())

`desc` has very high missingness (about 94.42%) and is excluded. `emp_title` and `title` are investigated further below before a decision is made.

In [ ]:
def investigate_text_columns(
    df,
    text_cols,
    target_col="loan_status",
    related_cols=None,
    top_n=20
):
    """
    Comprehensive investigation of text/free-text columns.

    Investigates:
    - Data type
    - Missingness
    - Empty strings
    - Raw cardinality
    - Normalized cardinality
    - Average / median / min / max text length
    - Rare-value concentration
    - Top-value coverage
    - Target availability
    - Relationship with target for common values
    - Relationship with related categorical columns

    Returns:
        summary_df
        analysis_results
    """

    if related_cols is None:
        related_cols = []

    summary = []
    analysis_results = {}

    for col in text_cols:

        s = df[col]

        # Basic investigation
        non_null = s.dropna().astype("string").str.strip()

        missing_count = s.isna().sum()
        missing_pct = s.isna().mean() * 100

        empty_count = non_null.eq("").sum()

        raw_unique = s.nunique(dropna=True)

        # Normalize text
        normalized = (
            non_null
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

        normalized_unique = normalized.nunique()

        # Text length
        lengths = normalized.str.len()

        # Frequency analysis
        value_counts = normalized.value_counts()

        unique_once = (value_counts == 1).sum()
        unique_once_pct = (
            unique_once / len(value_counts) * 100
            if len(value_counts) > 0 else 0
        )

        top_n_coverage = (
            value_counts.head(top_n).sum()
            / value_counts.sum()
            * 100
            if value_counts.sum() > 0 else 0
        )

        # Target relationship
        target_table = None

        if target_col in df.columns:

            valid_mask = normalized.notna() & df[target_col].notna()

            temp = pd.DataFrame({
                "text": normalized[valid_mask],
                "target": df.loc[valid_mask, target_col]
            })

            top_values = (
                temp["text"]
                .value_counts()
                .head(top_n)
                .index
            )

            target_table = pd.crosstab(
                temp.loc[temp["text"].isin(top_values), "text"],
                temp.loc[temp["text"].isin(top_values), "target"],
                normalize="index"
            ).round(4)

        # Related-column analysis
        related_analysis = {}

        for related_col in related_cols:

            if related_col not in df.columns:
                continue

            mask = normalized.notna() & df[related_col].notna()

            temp = pd.DataFrame({
                "text": normalized[mask],
                "related": df.loc[mask, related_col]
            })

            top_values = (
                temp["text"]
                .value_counts()
                .head(top_n)
                .index
            )

            related_analysis[related_col] = pd.crosstab(
                temp.loc[temp["text"].isin(top_values), "text"],
                temp.loc[temp["text"].isin(top_values), "related"],
                normalize="index"
            ).round(4)

        # Store summary
        summary.append({
            "Column": col,
            "Data_Type": str(s.dtype),

            "Rows": len(df),

            "Missing_Count": missing_count,
            "Missing_%": missing_pct,

            "Empty_String_Count": empty_count,
            "Empty_String_%": empty_count / len(df) * 100,

            "Raw_Unique": raw_unique,
            "Raw_Unique_%": raw_unique / len(df) * 100,

            "Normalized_Unique": normalized_unique,
            "Normalized_Unique_%": normalized_unique / len(df) * 100,

            "Avg_Text_Length": lengths.mean(),
            "Median_Text_Length": lengths.median(),
            "Min_Text_Length": lengths.min(),
            "Max_Text_Length": lengths.max(),

            "Unique_Values_Occurring_Once": unique_once,
            "Unique_Values_Occurring_Once_%": unique_once_pct,

            f"Top_{top_n}_Coverage_%": top_n_coverage
        })

        analysis_results[col] = {
            "top_values": value_counts.head(top_n),
            "target_distribution": target_table,
            "related_columns": related_analysis
        }

    # Final summary DataFrame
    summary_df = pd.DataFrame(summary)

    summary_df = summary_df.round(2)

    # Print final investigation
    print("\n" + "=" * 80)
    print("TEXT COLUMN INVESTIGATION SUMMARY")
    print("=" * 80)

    display(summary_df)

    for col in text_cols:

        print("\n" + "=" * 80)
        print(f"DETAILED INVESTIGATION: {col}")
        print("=" * 80)

        print("\nTop Values:")
        display(analysis_results[col]["top_values"])

        if analysis_results[col]["target_distribution"] is not None:
            print("\nTarget Distribution of Top Values:")
            display(
                analysis_results[col]["target_distribution"]
            )

        for related_col, table in (
            analysis_results[col]["related_columns"].items()
        ):
            print(f"\nRelationship with {related_col}:")
            display(table)

    return summary_df, analysis_results

In [ ]:
text_summary, text_analysis = investigate_text_columns(
    df=df,
    text_cols=["emp_title", "title"],
    target_col="loan_status",
    related_cols=["emp_length", "purpose"],
    top_n=20,
)

- **`title`:** low missingness but high raw cardinality. Most values are highly repetitive and closely aligned with `purpose`, so the column adds largely redundant information. It is excluded from the current feature set. Text-based feature extraction may be considered in a later experiment.
- **`emp_title`:** the loan-status distribution differs among frequently occurring titles, so the column may carry predictive information. However, its very high cardinality and large share of rare values make direct categorical encoding impractical. Using it would require text normalisation, occupation grouping or NLP-based feature engineering. It is excluded from the baseline feature set and may be evaluated in a later feature-engineering experiment.

| Column | Final decision | Reason |
| --- | --- | --- |
| `emp_title` | **Exclude from current model** | 7.39% missing, about 410K normalised unique values, and 74.31% of unique values occur only once. It may contain useful information, but direct categorical encoding is impractical and dedicated NLP or occupation grouping would be required. |
| `desc` | **Exclude** | 94.42% missing and highly unstructured free text. |
| `title` | **Exclude** | Low missingness, but largely redundant with `purpose`; the most common titles map almost directly to purpose categories. |

**Decision:** `emp_title`, `desc` and `title` are excluded from the current feature set.

## 9. Final Column Classification

Decisions are applied to the categories defined in Section 6, so that every column has a single, consistent classification.

In [ ]:
category_decisions = {
    "Application Information": "Include as candidate feature",
    "Credit History": "Include as candidate feature",
    "Joint Application / Credit History": "Include as candidate feature (joint applications only)",
    "Lender-Assigned / Listing Attributes": "Exclude from feature set",
    "Funding / Loan Issuance": "Exclude from feature set",
    "Post-Loan Performance": "Exclude from feature set",
    "Hardship / Payment Modification": "Exclude from feature set",
    "Debt Settlement": "Exclude from feature set",
    "Identifiers / Administrative": "Exclude from feature set",
    "Text / Free Text": "Exclude from feature set",
    "Target": "Target (not a feature)",
}

classification_df = pd.DataFrame(
    [
        {"Column": col, "Category": category, "Decision": category_decisions[category]}
        for category, columns in column_categories.items()
        for col in columns
    ]
)

include_df = classification_df[classification_df["Decision"].str.startswith("Include")].reset_index(drop=True)
exclude_df = classification_df[~classification_df["Decision"].str.startswith("Include")].reset_index(drop=True)

In [ ]:
display(include_df)
display(exclude_df)

### Classification check

In [ ]:
classified = classification_df["Column"]

duplicated_columns = classified[classified.duplicated()].tolist()
missing_columns = set(df.columns) - set(classified)
extra_columns = set(classified) - set(df.columns)

print(f"Original columns   : {df.shape[1]}")
print(f"Included columns   : {len(include_df)}")
print(f"Excluded columns   : {len(exclude_df)}")
print(f"Total classified   : {classified.nunique()}")
print(f"Duplicated entries : {duplicated_columns}")
print(f"Missing from classification   : {missing_columns}")
print(f"Not present in the dataset    : {extra_columns}")

assert not duplicated_columns and not missing_columns and not extra_columns

## 10. Summary and Next Step: Data Cleaning

### Findings
- The dataset has 2,260,701 rows and 151 columns. Of these, 96 columns are candidate features, 54 are excluded because they are identifiers, free text, lender-assigned or generated after the application, and `loan_status` is the target.
- Missingness in the hardship, settlement and joint-application columns is structural ("not applicable"), not random.
- Only 34 rows have missing values in the low-missingness categorical columns. `emp_length` has 6.5% missing values.

### Cleaning plan (notebook 03)
1. **Remove invalid rows.** Inspect the 34 rows with missing categorical values, then drop them. Check for fully duplicated rows.
2. **Define the target.** Convert `loan_status` into a binary default indicator. Remove loans without a known outcome (for example "Current") and record how each status is mapped.
3. **Apply the column selection.** Keep the candidate features, the target and `issue_d` (needed only for time-based splitting, not as a feature).
4. **Fix data types.** Convert `term` to a number, `emp_length` to an ordered numeric value with a separate missing category, the credit-line dates to dates, and `zip_code` to a text field.
5. **Assess missingness in numeric candidates.** Separate structural from random missingness, for example columns introduced only for later loans, and decide between a missing-value indicator, imputation or removal for each group.
6. **Handle joint applications.** Create an application-type indicator and combine the joint and individual income and DTI values.
7. **Check value validity.** Look for impossible or implausible values (for example in `dti`, `annual_inc`, `revol_util`, and FICO low above high) and decide on correction or capping after the EDA.
8. **Clean categorical fields.** Standardise text values and decide how to treat rare categories and high-cardinality fields (`addr_state`, `zip_code`).
9. **Save the result.** Write the cleaned dataset to `data/interim/` as parquet and log the row and column counts after each step.

### Rule for the next stage
Split the data (time-based, using `issue_d`) **before** fitting any imputer, encoder or scaler. These steps must be learned on the training data only, to avoid leakage.